# 03 — Hybrid RAG eval + optional publish

Phase 5 path: **clone → corpus → index → retrieval metrics → optional train → grounded generate eval → publish**.

**Defaults are dry-run / CPU-safe.** Do not flip `RUN_GPU` unless you are on a Kaggle T4 (or similar) and intend to download embed weights / load Unsloth.

Do **not** copy fixture Recall@k / nDCG / citation-hit percentages onto a resume. Those numbers live in `evals/reports/` after a real `--run`.

## 0. Flags (dry-run by default)

In [ ]:
from pathlib import Path
import os
import sys

# Flip only on Kaggle GPU after you have accepted the extra download / train time.
RUN_GPU = False
RUN_TRAIN = False          # short SFT smoke; requires RUN_GPU
RUN_GENERATE_EVAL = False # Unsloth generate; requires RUN_GPU
PUBLISH_ADAPTER = False   # HF Hub upload; needs HF_TOKEN secret + adapter dir

IN_KAGGLE = Path("/kaggle").exists()
print("IN_KAGGLE:", IN_KAGGLE)
print("RUN_GPU:", RUN_GPU, "RUN_TRAIN:", RUN_TRAIN, "RUN_GENERATE_EVAL:", RUN_GENERATE_EVAL, "PUBLISH_ADAPTER:", PUBLISH_ADAPTER)

## 1. Clone / path setup

In [ ]:
# If this notebook lives in the repo, use ../src. On a fresh Kaggle kernel, clone first.
REPO = Path("..").resolve()
if not (REPO / "src" / "earnings_call_research_assistant").exists():
    if IN_KAGGLE:
        !git clone https://github.com/nuwanda94/earnings-call-research-assistant.git
        REPO = Path("earnings-call-research-assistant").resolve()
    else:
        raise FileNotFoundError("Cannot find package src; run from notebooks/ or clone the repo.")

src = (REPO / "src").resolve()
if str(src) not in sys.path:
    sys.path.insert(0, str(src))
os.chdir(REPO)
print("cwd:", Path.cwd())
print("src:", src)

In [ ]:
if IN_KAGGLE:
    %pip install -q pyyaml
    if RUN_GPU:
        %pip install -q unsloth transformers accelerate bitsandbytes sentence-transformers

## 2. Corpus (defines N)

In [ ]:
!python scripts/build_rag_corpus.py

In [ ]:
import json
from pathlib import Path

manifest_path = Path("data/rag/corpus_v0.1.0/manifest.json")
if manifest_path.is_file():
    manifest = json.loads(manifest_path.read_text())
    print("N chunks:", manifest.get("n_chunks"))
    print("n_documents:", manifest.get("n_documents"))
    print("version:", manifest.get("version"))
    print("sources:", manifest.get("sources"))
else:
    print("manifest missing — corpus script should have written it")

## 3. Hybrid index (BM25 + dense)

Dry-run uses hashed bag-of-tokens embeddings (offline). `RUN_GPU` rebuilds dense with `sentence-transformers`.

In [ ]:
if RUN_GPU:
    !python scripts/build_rag_index.py --run --query "operating margin guidance" --k 5
else:
    !python scripts/build_rag_index.py --query "operating margin guidance" --k 3

## 4. Gold eval set (seed 3407)

In [ ]:
!python scripts/build_rag_eval_set.py

## 5. Retrieval metrics — Recall@k / nDCG@k

In [ ]:
if RUN_GPU:
    !python scripts/eval_retrieval.py --run
else:
    !python scripts/eval_retrieval.py

In [ ]:
metrics_path = Path("evals/reports/rag_metrics.json")
if metrics_path.is_file():
    payload = json.loads(metrics_path.read_text())
    print("n_queries:", payload.get("n_queries"))
    print("n_index_docs:", payload.get("n_index_docs"))
    print("dense_backend:", payload.get("dense_backend"))
    for name, report in (payload.get("backends") or {}).items():
        print(name, "recall", report.get("mean_recall"), "ndcg", report.get("mean_ndcg"))
    print("note: fixture / dry-run scores are not resume numbers")
else:
    print("rag_metrics.json not written")

## 6. Optional short SFT (off by default)

Leave `RUN_TRAIN=False` in automation and most sessions. On Kaggle T4 set both `RUN_GPU` and `RUN_TRAIN` for a 20-step smoke.

In [ ]:
if RUN_TRAIN and RUN_GPU:
    !python scripts/train_sft.py --run --max-steps 20
else:
    !python scripts/train_sft.py
    print("SFT dry-run only (no weights). Set RUN_TRAIN and RUN_GPU for a short GPU smoke.")

## 7. Grounded generate eval (citation-hit + token F1)

Dry-run writes placeholders. `--run` loads `InferenceHarness` (base + optional adapter).

In [ ]:
ADAPTER = Path("outputs/adapters/llama32-3b-ecra-sft")
if RUN_GENERATE_EVAL and RUN_GPU:
    if ADAPTER.is_dir():
        !python scripts/eval_rag_generate.py --run --adapter-dir outputs/adapters/llama32-3b-ecra-sft
    else:
        !python scripts/eval_rag_generate.py --run
        print("adapter dir missing; scored base only")
else:
    !python scripts/eval_rag_generate.py

In [ ]:
gen_path = Path("evals/reports/rag_generation_metrics.json")
if gen_path.is_file():
    gen = json.loads(gen_path.read_text())
    print("dry_run:", gen.get("dry_run"))
    print("n_queries:", gen.get("n_queries"))
    print("top_k:", gen.get("top_k"))
    print("adapter_dir:", gen.get("adapter_dir"))
    print("aggregate:", json.dumps(gen.get("aggregate"), indent=2))
    print(gen.get("note"))
else:
    print("rag_generation_metrics.json not written")

## 8. Publish adapter (optional, token via env only)

Never paste `HF_TOKEN` into a cell. On Kaggle: Add-ons → Secrets → `HF_TOKEN`.

In [ ]:
if IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
        if token:
            os.environ["HF_TOKEN"] = token
            print("HF_TOKEN loaded from Kaggle secrets (value not printed)")
        else:
            print("HF_TOKEN secret empty")
    except Exception as exc:
        print("no Kaggle secret:", type(exc).__name__)

token_set = bool(os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN"))
print("token_env_set:", token_set)

if PUBLISH_ADAPTER and token_set and ADAPTER.is_dir():
    !python scripts/publish_adapter.py --repo-id nuwanda94/llama32-3b-ecra-sft --run
else:
    !python scripts/publish_adapter.py
    print("publish dry-run only")

## 9. Done

Artifacts to commit after a **human** Kaggle `--run` (not this automation):

- `data/rag/corpus_v0.1.0/manifest.json` — measured **N**
- `evals/reports/rag_metrics.json` — Recall@k / nDCG@k
- `evals/reports/rag_generation_metrics.json` — citation-hit + token F1

Next automation item: `evals/reports/RAG_EVAL_REPORT.md` + README results + `docs/MODEL_CARD_RAG.md` (Phase 5.8). Fill numbers **only** from those JSON files.